# 10. TabNet — pytorch-tabnet on CUDA (FOC-175, phase F3)

The F3 question, third model family: **does a deep tabular architecture with
instance-wise feature selection (TabNet) beat the tree arms on this fraud
problem — and does anything survive the customer-disjoint axes that collapsed
every arm so far?**

Where the ladder stands (docs/FOC-174-report.md, nb7/nb8/nb9):

- **F0** — XGB baseline on the 10 transaction features: test PR-AUC 0.0532
  (chronological split).
- **F1** — base + client features (nb7 arm b): 0.2374 chronological — read
  there as customer-identity signal, not transferable demographics.
- **F2** — SCE 0.0340 / dictionary 0.1034 chronological (nb8); on the
  customer-grouped splits every arm collapsed to ~0.01, next to chance.
- **F3 so far** — gbdt-ensemble (nb9): 0.0145 random-grouped, i.e. chance-level
  on the primary axis too; only the chronological axis (24 test positives,
  identity signal in play) separates from chance.
- **F3 primary axis** — random-grouped (FOC-175): customers assigned to
  train/test by a seeded random draw — customer-disjoint, no time ordering.

The arm (`tabnet` in the unified runner `src/fraud_pipeline.py`, implemented in
`src/arms_tabnet.py`): TabNetClassifier (pytorch-tabnet 4.1.0) on the SAME
116-column base+client feature matrix as the xgb-client / gbdt-ensemble arms
(LightGBM-safe column names shared via `arms_gbdt.sanitize_feature_names`),
library-default architecture (n_d=n_a=8, 3 decision steps, sparsemax masks,
Adam lr 2e-2), CUDA when available else CPU, fixed seeds with
`cudnn.deterministic=True` (GPU determinism attempted, not guaranteed — the
observed behavior is measured and reported below).

Two house flags, stated upfront and unpacked in the interpretation: (1)
**pytorch-tabnet is in stalled maintenance** (last release 4.1.0); (2) **TabNet
has weak native imbalance handling** — no `scale_pos_weight`-style loss weight;
the only documented classifier knob is `fit(weights=1)`, an inverse-frequency
`WeightedRandomSampler` (library-managed minority oversampling). This arm uses
exactly that documented knob — nothing beyond it (no SMOTE, no manual
resampling) — and the frozen-threshold tuning on the runner's validation carve
stays the threshold-side compensator.

Protocol (nb7/nb8/nb9 discipline, driven through the runner API — never
re-implemented here): one split per axis, stratified validation carve from
TRAIN only, threshold frozen on the carve, one-shot frozen-threshold test
evaluation with percentile-bootstrap AUC intervals. Every results table below
carries test positives and the chance level (the test positive rate a random
ranking lands at) — 91 frauds total make one unreadable without the other.

In [1]:
# Runtime provenance - executed in the phase worktree venv built from the
# pinned requirements (kernel python3). Printed so the committed, executed
# notebook self-documents the exact runtime the numbers were produced on.
import importlib.metadata
import platform
import sys

import numpy
import pandas
import sklearn
import torch

print('python:', sys.version.split()[0], '| platform:', platform.platform())
print('kernel: python3 (nbclient + WindowsSelectorEventLoopPolicy)')
for _mod in (pandas, numpy, sklearn, torch):
    print('%s: %s' % (_mod.__name__, _mod.__version__))
print('pytorch-tabnet:', importlib.metadata.version('pytorch-tabnet'))
print('torch cuda available:', torch.cuda.is_available())
print('cuda device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu only')

python: 3.11.9 | platform: Windows-10-10.0.26200-SP0
kernel: python3 (nbclient + WindowsSelectorEventLoopPolicy)
pandas: 2.3.3
numpy: 2.4.6
sklearn: 1.7.2
torch: 2.11.0+cu128
pytorch-tabnet: 4.1.0


torch cuda available: True
cuda device: NVIDIA GeForce RTX 5070 Ti Laptop GPU


## The arm through the runner — all three axes

`run_arm_on_axis` owns the whole protocol per axis: axis split → validation
carve → model → fit (early stopping on the arm's internal stratified carve of
the fitting rows) → threshold frozen on the runner's carve → test metrics +
1000-sample percentile-bootstrap AUC CIs. `cv=False` here and the arm ships
`supports_cv=False` ([no-cv]): per-fold GPU clones would each re-carve and
re-run early stopping for little added evidence at this scale. The axes, in
registry order:

- **random-grouped (PRIMARY)** — seeded random customer assignment,
  customer-disjoint, no time ordering;
- **grouped (stress)** — test = latest-seen customers;
- **chronological (stress)** — test strictly later than train.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from arms_tabnet import make_tabnet
from fraud_pipeline import (
    ARMS,
    AXES,
    BASE_FEATURES,
    CLIENT_CATEGORICAL,
    CLIENT_NUMERIC,
    DEFAULT_RESULTS_PATH,
    axis_split,
    best_f1_threshold,
    load_enriched,
    load_results,
    print_comparison_table,
    rich_test_metrics,
    run_arm_on_axis,
)

# Canonical enriched frame + labels from the unified runner (the nb7/nb8 data
# section, shared by every arm — loaded once, used by everything below).
enriched, y = load_enriched()
print(
    'fraud txns: %d of %d (%.2f%%) across %d unique customers'
    % (int(y.sum()), len(y), 100 * y.mean(), enriched['customer'].nunique())
)

fraud txns: 91 of 5302 (1.72%) across 100 unique customers


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-175-f3\src\funs.py:197: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  trxns_data["timestamp"] = pd.to_datetime(


In [3]:
rows = []
for axis in AXES:  # registry order: random-grouped (PRIMARY), grouped, chronological
    rows.append(run_arm_on_axis('tabnet', axis, enriched, y, cv=False))
print_comparison_table(rows, title='tabnet — test metrics per axis (frozen threshold)')


Early stopping occurred at epoch 31 with best_epoch = 21 and best_val_auc = 0.8495


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-175-f3\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 44 with best_epoch = 34 and best_val_auc = 0.87184


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-175-f3\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 15 with best_epoch = 5 and best_val_auc = 0.83461


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-175-f3\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



== tabnet — test metrics per axis (frozen threshold) ==
          axis    arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high     f1  recall_at_precision  frozen_threshold  n_features
random-grouped tabnet     ok              13        977        0.0133  0.0138         0.0077          0.0254   0.4999          0.3515           0.6264 0.0000                  0.0            0.9778         116
       grouped tabnet     ok              11        831        0.0132  0.0150         0.0075          0.0268   0.5516          0.4315           0.6584 0.0000                  0.0            0.5446         116
 chronological tabnet     ok              24       1061        0.0226  0.0245         0.0133          0.0729   0.4325          0.3357           0.5415 0.0526                  0.0            0.9674         116


## tabnet vs gbdt-ensemble — primary axis only

The F3 family question, head to head: TabNet against the GBDT soft-vote
ensemble (nb9) under the identical protocol on the identical split. The
gbdt-ensemble row comes from the accumulated results file
(`results/fraud_pipeline_results.jsonl`, latest row per axis x arm — the same
superseding convention the runner's table uses; if the row is missing the cell
re-runs the arm through the pipeline instead of guessing). Both rows carry
their test positive count, chance level and bootstrap CIs — with 13 test
positives the honest benchmark is distance-from-chance plus interval overlap,
not the point estimate alone.

In [4]:
def latest_ok(axis, arm):
    # Latest ok row per (axis, arm) from the accumulated JSONL.
    matches = [
        r
        for r in load_results(DEFAULT_RESULTS_PATH)
        if r.get('axis') == axis and r.get('arm') == arm and r.get('status') == 'ok'
    ]
    return matches[-1] if matches else None


gbdt_row = latest_ok('random-grouped', 'gbdt-ensemble')
if gbdt_row is None:  # results file unavailable/stale — re-run through the pipeline
    gbdt_row = run_arm_on_axis('gbdt-ensemble', 'random-grouped', enriched, y, cv=False)
tabnet_row = next(r for r in rows if r['axis'] == 'random-grouped')

print_comparison_table(
    [gbdt_row, tabnet_row],
    title='tabnet vs gbdt-ensemble — random-grouped (PRIMARY axis)',
)


def covers_chance(row):
    return row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']


print(
    'test positives: %d | chance PR-AUC %.4f | tabnet %.4f [CI %.4f, %.4f] | '
    'gbdt-ensemble %.4f [CI %.4f, %.4f] | delta %+.4f'
    % (
        tabnet_row['test_positives'],
        tabnet_row['chance_level'],
        tabnet_row['pr_auc'], tabnet_row['pr_auc_ci_low'], tabnet_row['pr_auc_ci_high'],
        gbdt_row['pr_auc'], gbdt_row['pr_auc_ci_low'], gbdt_row['pr_auc_ci_high'],
        tabnet_row['pr_auc'] - gbdt_row['pr_auc'],
    )
)
print(
    'CI covers chance: tabnet %s | gbdt-ensemble %s -> %s'
    % (
        covers_chance(tabnet_row),
        covers_chance(gbdt_row),
        'both arms indistinguishable from a random ranking on this axis'
        if covers_chance(tabnet_row) and covers_chance(gbdt_row)
        else 'at least one arm separates from chance (read the intervals above)',
    )
)


== tabnet vs gbdt-ensemble — random-grouped (PRIMARY axis) ==
          axis           arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high  f1  recall_at_precision  frozen_threshold  cv_pr_auc_mean  cv_pr_auc_std  n_features
random-grouped gbdt-ensemble     ok              13        977        0.0133  0.0145         0.0079          0.0343   0.4941          0.3560           0.6367 0.0                  0.0            0.3256          0.7132         0.1086         116
random-grouped        tabnet     ok              13        977        0.0133  0.0138         0.0077          0.0254   0.4999          0.3515           0.6264 0.0                  0.0            0.9778             NaN            NaN         116
test positives: 13 | chance PR-AUC 0.0133 | tabnet 0.0138 [CI 0.0077, 0.0254] | gbdt-ensemble 0.0145 [CI 0.0079, 0.0343] | delta -0.0007
CI covers chance: tabnet True | gbdt-ensemble True -> both arms indi

## Protocol refit, early-stopping evidence, determinism

`run_arm_on_axis` fits internally and discards the model, so the fitted model
is rebuilt here under the exact primary-axis protocol (same split, same carve,
same seeds) for the interpretability and determinism cells. Three reads off
this fit:

- **Early-stopping carve.** The arm cuts its own stratified 25% carve from the
  fitting rows (seed 42 — the same shape as the runner's threshold carve). That
  carve is customer-MIXED, unlike every test axis: a high carve val-AUC next to
  chance-level customer-disjoint test metrics is the nb7/nb8/nb9
  identity-memorization signature in miniature (read below).
- **Sanity.** The refit's frozen-threshold test metrics must mirror the
  runner's random-grouped row from the table above.
- **Determinism.** Two additional fits under the same seeds; the max absolute
  probability deviation is printed and reported honestly — `cudnn.deterministic`
  is an attempt, not a guarantee, and the committed outputs state what was
  actually observed.

In [5]:
X_raw = ARMS['tabnet']['build_features'](enriched)
train_idx, test_idx = axis_split('random-grouped', enriched, y)
X_tr, X_te = X_raw.loc[train_idx], X_raw.loc[test_idx]
y_tr, y_te = y.loc[train_idx], y.loc[test_idx]
X_fit, X_val, y_fit, y_val = train_test_split(
    X_tr, y_tr, test_size=0.25, random_state=42, stratify=y_tr
)

fit_model = make_tabnet(y_fit).fit(X_fit, y_fit)
carve = fit_model.carve_
val_hist = fit_model.val_auc_history_
print(
    'internal carve: %d fit rows (%d positives) / %d val rows (%d positives) | device %s'
    % (
        carve['fit_rows'], carve['fit_positives'], carve['val_rows'],
        carve['val_positives'], fit_model.device_name_,
    )
)
print(
    'early stopping: best val AUC %.4f at epoch %d of %d run (patience on val AUC)'
    % (max(val_hist), int(np.argmax(val_hist)) + 1, len(val_hist))
)

# Sanity: identical protocol, identical seeds -> must mirror the runner row.
threshold = best_f1_threshold(y_val, fit_model.predict_proba(X_val)[:, 1])
proba_a = fit_model.predict_proba(X_te)[:, 1]
refit_metrics = rich_test_metrics(y_te, proba_a, threshold)
print(
    'refit sanity: test PR-AUC %.4f vs runner row %.4f | F1 %.4f | threshold %.4f'
    % (refit_metrics['pr_auc'], tabnet_row['pr_auc'], refit_metrics['f1'], threshold)
)

# Determinism: two more fits, same seeds, same data -> measure the deviation.
proba_b = make_tabnet(y_fit).fit(X_fit, y_fit).predict_proba(X_te)[:, 1]
proba_c = make_tabnet(y_fit).fit(X_fit, y_fit).predict_proba(X_te)[:, 1]
max_dev = float(max(np.max(np.abs(proba_a - proba_b)), np.max(np.abs(proba_b - proba_c))))
print(
    'within-kernel refit determinism (3 fits, same seeds): max |delta proba| = %.3e -> %s'
    % (max_dev, 'bit-identical predictions' if max_dev == 0.0 else 'GPU nondeterminism observed')
)


Early stopping occurred at epoch 31 with best_epoch = 21 and best_val_auc = 0.8495
internal carve: 2432 fit rows (43 positives) / 811 val rows (15 positives) | device cuda
early stopping: best val AUC 0.8495 at epoch 22 of 32 run (patience on val AUC)
refit sanity: test PR-AUC 0.0138 vs runner row 0.0138 | F1 0.0000 | threshold 0.9778


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-175-f3\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 31 with best_epoch = 21 and best_val_auc = 0.8495


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-175-f3\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 31 with best_epoch = 21 and best_val_auc = 0.8495
within-kernel refit determinism (3 fits, same seeds): max |delta proba| = 0.000e+00 -> bit-identical predictions


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-175-f3\.venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [6]:
# Attention-mask aggregation by feature group, over the FITTING rows only:
# this describes what the model fit, not a validated fraud signal (the test
# axes are at/near chance — see interpretation).
M_explain, masks = fit_model.explain_matrix(X_fit, normalize=True)
names = fit_model.feature_names_


def group_of(name):
    # get_dummies names dummies f'{col}_{category}', so an exact match or a
    # full-column prefix recovers the originating column; the two client
    # numerics pass through under their own names.
    for col in BASE_FEATURES:
        if name == col or name.startswith(col + '_'):
            return 'base categorical'
    for col in CLIENT_CATEGORICAL:
        if name == col or name.startswith(col + '_'):
            return 'client categorical'
    if name in CLIENT_NUMERIC:
        return 'client numeric'
    return 'other'


groups = np.array([group_of(n) for n in names])
share = M_explain.mean(axis=0)  # rows normalized -> mean share per feature, sums to 1
group_share = (
    pd.DataFrame({'group': groups, 'share': share})
    .groupby('group')['share']
    .sum()
    .sort_values(ascending=False)
)
print('mask share by feature group (fit rows, sums to 1):')
print((group_share * 100).round(2).to_string())

top = (
    pd.DataFrame({'feature': names, 'group': groups, 'share': share})
    .sort_values('share', ascending=False)
    .head(8)
)
print('\ntop-8 features by mean mask share:')
print(top.round(4).to_string(index=False))

# The literal per-decision-step attention masks, row-normalized per step.
step_rows = []
for step in sorted(masks):
    m = np.asarray(masks[step], dtype=float)
    m = m / np.maximum(m.sum(axis=1, keepdims=True), 1e-12)
    per_group = pd.DataFrame({'group': groups, 'share': m.mean(axis=0)}).groupby('group')['share'].sum()
    step_rows.append({'step': step, **{g: float(v) for g, v in per_group.items()}})
print('\nper-decision-step mask share by group (rows normalized per step):')
print(pd.DataFrame(step_rows).set_index('step').round(3).to_string())

mask share by feature group (fit rows, sums to 1):
group
base categorical      73.03
client categorical    25.21
client numeric         1.75

top-8 features by mean mask share:
                                feature              group  share
                                hour_11   base categorical 0.1254
                     channel_mobile_app client categorical 0.0875
                  income_band_3_40k_60k client categorical 0.0571
                      weekday_Wednesday   base categorical 0.0508
                                ccy_USD   base categorical 0.0455
         employment_industry_healthcare client categorical 0.0446
                counterparty_country_CN   base categorical 0.0438
amount_eur_bucket_(42443.123_ 57035.141   base categorical 0.0417

per-decision-step mask share by group (rows normalized per step):
      base categorical  client categorical  client numeric
step                                                      
0                0.793               0.201  

### Interpretation (read after the tables — null results are findings)

- **Noise budget first.** 91 frauds total; the three splits put 13
  (random-grouped), 11 (grouped) and 24 (chronological) test positives in play
  — printed on every table row next to its chance level. At that size a single
  swapped fraud moves test PR-AUC by hundredths and the bootstrap CIs span a
  wide band around every point estimate: deltas under ~0.05 PR-AUC between
  arms are noise, not signal.
- **Read every number against its chance level.** Chance PR-AUC equals the
  test positive rate: 0.0133 on random-grouped, 0.0132 on grouped, 0.0226 on
  chronological. The tabnet rows (first table) and the gbdt-ensemble
  comparison (second table) must be judged by their distance from those
  baselines AND by their printed CIs — an interval that still covers the
  chance level means the model is indistinguishable from a random ranking on
  that split, whatever the point estimate suggests. A null result here is a
  finding, not a failure: it says a deep tabular net with instance-wise
  feature selection extracts nothing the trees did not, at this data size.
- **Axis structure.** random-grouped is PRIMARY (customer-disjoint but
  temporally unbiased); grouped and chronological are stress tests that also
  remove recency overlap. nb7/nb8/nb9 showed the client-feature lift is
  customer-identity signal that collapses once test customers are disjoint
  from train; tabnet consumes the same feature set, so it inherits that
  collapse — compare its PR-AUC per axis against the chance levels above.
- **CV-vs-test identity-memorization signature.** This arm ships `supports_cv`
  = False ([no-cv]), so no CV summary is collected; the closest analogue is
  the arm's internal early-stopping carve (printed above): customer-MIXED
  rows, like the nb9 CV folds. If that carve's val AUC lands well above the
  customer-disjoint test numbers, that is the nb7/nb8/nb9 signature in
  miniature — the model memorizes customer identity available in training
  and it does not transfer across the customer boundary. Read the gap as
  memorization evidence, not as a bug in the protocol.
- **Mask importances are train-side descriptions.** The `explain()` masks
  above are aggregated over the FITTING rows. With test signal at ~null on
  the customer-disjoint axes, they describe what the model fit — which
  features it leaned on while training — not a validated fraud signal. Any
  'client numeric dominates' reading is a statement about memorizable
  identity features, consistent with the ladder's history, not evidence of
  a deployable detector.
- **Flag 1 — stalled maintenance.** pytorch-tabnet's last release is 4.1.0
  (the pinned version here); the project is in stalled maintenance. Any bug,
  CUDA-compat issue or API gap discovered downstream is unlikely to be fixed
  upstream — weight this arm's infrastructure cost accordingly if it ever
  earned a production case.
- **Flag 2 — weak native imbalance handling.** pytorch-tabnet has no
  `scale_pos_weight`-style loss weight. Its only documented classifier knob
  is `fit(weights=1)`: an inverse-frequency `WeightedRandomSampler` —
  library-managed minority oversampling, used here and disclosed (nothing
  beyond it: no SMOTE, no manual resampling). The frozen-threshold tuning on
  the validation carve is therefore the only threshold-side compensator, and
  it is what the reported F1 / recall-at-precision numbers rest on.
- **Determinism.** Seeds 42 everywhere (splits, carve, torch, numpy,
  sampler), `cudnn.deterministic=True` with `cudnn.benchmark=False` — an
  attempt, not a guarantee. Observed verdict, reported rather than smoothed
  over: three identical fits inside one kernel produced bit-identical test
  probabilities (max |delta| = 0.000e+00, printed above) and a full second
  execution of this notebook reproduced every printed number exactly - the
  committed outputs are CUDA-stable at this scale.

## Summary

- `src/arms_tabnet.py` implements the F3 `tabnet` arm: TabNetClassifier
  (pytorch-tabnet 4.1.0) on the shared base+client feature matrix, library-
  default architecture, CUDA when available else CPU, fixed seeds with
  cuDNN determinism requested, early stopping on an internal stratified
  carve, imbalance handled ONLY by the library's documented inverse-frequency
  sampler (`weights=1`).
- The arm is registered in the unified runner (`src/fraud_pipeline.py`) with
  `supports_cv=False` and is exercised end-to-end here through
  `run_arm_on_axis` on all three axes; the primary-axis row is benchmarked
  against the gbdt-ensemble arm from the accumulated results file.
- The verdicts are read from the tables above against their chance levels and
  bootstrap CIs; the committed CLI run of the arm appends the same protocol's
  numbers to `results/fraud_pipeline_results.jsonl`.
- Two report flags travel with this arm: pytorch-tabnet is in stalled
  maintenance (last release 4.1.0), and TabNet's native imbalance handling is
  weak — the frozen-threshold validation tuning is the only compensator
  beyond the disclosed sampler.